# Day 17: Triton Vector Add 与 RMSNorm

program_id 指向一块工作，arange 构建块内 offset，mask 处理尾部。RMSNorm 每个 program 处理一行，FP32 累积后恢复输出 dtype。

参考源码：`learning/kernels/triton_ops.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY15_21.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 17完整讲义](../docs/DAY17_TRITON_ROWS.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

Vector Add的N=1003、BLOCK=256：

```text
program0 → offsets 0..255
program1 → offsets 256..511
program2 → offsets 512..767
program3 → offsets 768..1023，只存到1002
```

一个Triton program不等于一个CUDA thread。
行宽127做RMSNorm时，载入128个lane形状，最后一个补0，
**均方分母仍是127，不是128**，最终只写127个有效值。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(17, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

BLOCK 大于行宽时 reduction 的补充值应该是什么？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：program等于thread吗？**
答：不是，它描述tile计算，底层线程/warp映射由编译器与配置决定。

**问：127补到128后，为什么不能除128？**
答：补位只是计算布局，不是多了一个真实输入特征。

**问：输入有requires_grad怎么办？**
答：当前wrapper拒绝；要训练需独立backward/autograd设计，不能直接detach掩盖需求。

**问：第一次调用很慢说明kernel很慢吗？**
答：可能包含JIT等开销，应区分启动与预热后稳态。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY17_TRITON_ROWS.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。